# Awareness Objective: Completed-Cycle Learning Notebook

**Business job:** Create visibility and measurable response before expecting a sale.

This notebook follows the objective from raw Meta and WhatsApp evidence through
objective-specific KPIs, Empirical Bayes with an explicit benchmark hierarchy,
efficiency, deterministic action, budget allocation, child-entity evidence, and
conversation diagnostics.

It is an explanatory report for a completed cycle. It does not optimize a live
campaign and does not ask an LLM to calculate scores or budget.

## 1. Scope And Reading Rules

1. `objective` is the only campaign grouping used for evaluation.
2. Paid-attributed WhatsApp outcomes are treated as complete for this MVP.
3. Organic/direct conversations are outside paid campaign scoring.
4. Active and stuck-pending outcomes remain visible but are excluded from mature rates.
5. The primary score measures outcome effectiveness; efficiency remains separate.
6. Small samples lean toward selected peers and always show a range. Same-objective
   peers are preferred; Awareness and Engagement may share Link CTR evidence when
   their own objective has too few peers.
7. Conversation signals create a separate quality score and can prioritize exploration.
   Primary outcome scores and final actions remain driven by structured evidence.
8. Budget is assigned only to campaigns; child levels guide execution without
   double-counting the same money.

In [1]:
from pathlib import Path
import sys

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(path for path in candidates if (path / "src_mvp").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src_mvp.config import load_budget_policy, load_objectives
from src_mvp.pipeline import run_mvp
from src_mvp.statistics import fit_beta_prior

FOCUS_OBJECTIVE = "OUTCOME_AWARENESS"
result = run_mvp(output_directory=ROOT / "src_mvp" / "outputs")
registry = load_objectives()
policy = load_budget_policy()
contract = registry.objectives[FOCUS_OBJECTIVE]
all_campaigns = result.scorecards["campaign"].copy()
campaigns = all_campaigns.loc[
    all_campaigns["objective"].eq(FOCUS_OBJECTIVE)
].copy()
allocations = {item.campaign_id: item for item in result.allocations}
objective_tests = [
    item for item in result.exploration_tests
    if item.objective == FOCUS_OBJECTIVE
]

DECISION_COLORS = {
    "scale": "#16815d",
    "hold": "#d68c16",
    "kill": "#c44536",
    "insufficient_evidence": "#6b7280",
}
alt.data_transformers.disable_max_rows()

print(f"Repository: {ROOT}")
print(f"Objective: {FOCUS_OBJECTIVE}")
print(f"Campaigns in scope: {len(campaigns)}")

Repository: /Users/abdelmoo/Desktop/CAPI Analysis/Team2-MarketingExpert
Objective: OUTCOME_AWARENESS
Campaigns in scope: 2


## 2. Objective Contract

The objective determines the business question and the one primary statistical score.
It also selects one efficiency metric, which remains a separate gate.

**Important limitation:** Link click-through rate is an available response proxy. It does not prove brand awareness lift, which would require a brand-lift study or comparable survey.

In [2]:
contract_table = pd.DataFrame([
    ["Business job", contract.business_job],
    ["Success question", contract.success_question],
    ["Primary metric", contract.primary_metric],
    ["Numerator", contract.primary_numerator],
    ["Denominator", contract.primary_denominator],
    ["Better primary direction", contract.primary_direction],
    ["Efficiency metric", contract.efficiency_metric],
    ["Better efficiency direction", contract.efficiency_direction],
    ["Minimum primary trials", contract.minimum_trials],
    ["Minimum peer entities", contract.minimum_peer_entities],
    ["Configured fallback group", contract.fallback_benchmark_group or "None"],
    ["Semantic metric", contract.semantic.metric],
    ["Semantic definition", contract.semantic.definition],
], columns=["Contract element", "Value"])
display(contract_table.style.hide(axis="index"))

Contract element,Value
Business job,Create efficient visibility and measurable response.
Success question,Did the campaign create more link response than comparable awareness campaigns?
Primary metric,link_ctr
Numerator,link_clicks
Denominator,impressions
Better primary direction,higher
Efficiency metric,cpm
Better efficiency direction,lower
Minimum primary trials,10
Minimum peer entities,2


## 3. Campaign Inventory And Delivery

Spend and delivery volume describe campaign size; they do not prove success. The
primary outcome score below evaluates quality relative to the selected compatible
benchmark. Portfolio-wide context remains descriptive only.

In [3]:
inventory = campaigns[[
    "campaign_name", "entity_status", "media_start", "media_end",
    "running_days", "active_days", "spend", "impressions", "link_clicks",
    "observed_conversations", "mature_conversations",
    "unresolved_conversations",
]].rename(columns={
    "campaign_name": "Campaign", "entity_status": "Status",
    "media_start": "First delivery", "media_end": "Last delivery",
    "running_days": "Calendar days", "active_days": "Days with delivery",
    "spend": "Spend", "impressions": "Impressions",
    "link_clicks": "Link clicks",
    "observed_conversations": "Paid conversations",
    "mature_conversations": "Mature",
    "unresolved_conversations": "Unresolved",
})
display(inventory.style.format({"Spend": "{:,.2f}", "Impressions": "{:,.0f}"}))

,Campaign,Status,First delivery,Last delivery,Calendar days,Days with delivery,Spend,Impressions,Link clicks,Paid conversations,Mature,Unresolved
1,Awareness Boost January,COMPLETED,2026-01-15 00:00:00,2026-02-14 00:00:00,31,31,"5,966.66","1,089,878",13033,4,4,0
11,Summer Retention Push,COMPLETED,2026-06-05 00:00:00,2026-06-27 00:00:00,23,23,"9,006.65","1,806,267",22208,16,15,1


## 4. Raw Primary Evidence

The raw score is the directly observed rate:

```text
raw rate = objective successes / eligible objective trials
```

These counts remain beside every corrected score so the modeled result can always be
audited back to real evidence.

In [4]:
raw_evidence = campaigns[[
    "campaign_name", "score_successes", "score_trials", "raw_rate",
    "observed_conversations", "mature_conversations",
    "unresolved_conversations", "evidence_status",
]].rename(columns={
    "campaign_name": "Campaign", "score_successes": "Successes",
    "score_trials": "Eligible trials", "raw_rate": "Raw rate",
    "observed_conversations": "Observed conversations",
    "mature_conversations": "Mature conversations",
    "unresolved_conversations": "Unresolved conversations",
    "evidence_status": "Evidence status",
})
display(raw_evidence.style.format({"Raw rate": "{:.2%}"}, na_rep="Not available"))

maturity_long = inventory[["Campaign", "Mature", "Unresolved"]].melt(
    "Campaign", var_name="Outcome maturity", value_name="Conversations"
)
maturity_chart = alt.Chart(maturity_long).mark_bar().encode(
    y=alt.Y("Campaign:N", sort="-x", title=None),
    x=alt.X("Conversations:Q", title="WhatsApp conversations"),
    color=alt.Color(
        "Outcome maturity:N",
        scale=alt.Scale(domain=["Mature", "Unresolved"], range=["#218380", "#d9a441"]),
        legend=alt.Legend(orient="top"),
    ),
    tooltip=["Campaign", "Outcome maturity", "Conversations"],
).properties(width=760, height=max(120, len(campaigns) * 42))
display(maturity_chart)

,Campaign,Successes,Eligible trials,Raw rate,Observed conversations,Mature conversations,Unresolved conversations,Evidence status
1,Awareness Boost January,13033.000000,1089878.000000,1.20%,4,4,0,sufficient
11,Summer Retention Push,22208.000000,1806267.000000,1.23%,16,15,1,sufficient


alt.Chart(...)

## 5. Leave-One-Out Peer Evidence And Prior

The benchmark hierarchy is explicit:

1. Use other campaigns with the same objective when at least two are valid.
2. If those peers are insufficient, Awareness and Engagement may use their shared
   `upper_funnel_link_ctr` group because both have the same Link CTR numerator,
   denominator, and direction.
3. Mark this fallback `provisional` and cap the final action at `KEEP_AS_TEST`.
4. Show the all-portfolio same-metric rate only as context. It never enters the prior,
   corrected score, uncertainty range, or funding decision.

The campaign being scored is always removed from its own benchmark.

```text
adjusted peer center = (peer successes + 0.5) / (peer trials + 1)
prior strength = conservative equivalent evidence allowed from peers
prior alpha = peer center x prior strength
prior beta = (1 - peer center) x prior strength
```

The prior is learned from the selected peer evidence. It is not a business target.
At least two valid compatible peers are required; otherwise the result is insufficient
evidence.

In [5]:
prior_rows = []
for _, row in campaigns.iterrows():
    same_objective_peers = all_campaigns.loc[
        all_campaigns["objective"].eq(FOCUS_OBJECTIVE)
        & all_campaigns["entity_id"].ne(row["entity_id"])
    ].copy()
    compatible_objectives = [
        name for name, candidate in registry.objectives.items()
        if contract.fallback_benchmark_group
        and candidate.fallback_benchmark_group == contract.fallback_benchmark_group
        and candidate.primary_metric == contract.primary_metric
        and candidate.primary_numerator == contract.primary_numerator
        and candidate.primary_denominator == contract.primary_denominator
        and candidate.primary_direction == contract.primary_direction
    ]
    if row["benchmark_scope"] == "same_objective":
        selected_objectives = [FOCUS_OBJECTIVE]
    elif row["benchmark_scope"] == "shared_primary_kpi_group":
        selected_objectives = compatible_objectives
    else:
        selected_objectives = []
    peers = all_campaigns.loc[
        all_campaigns["objective"].isin(selected_objectives)
        & all_campaigns["entity_id"].ne(row["entity_id"])
    ].copy()
    for frame in [same_objective_peers, peers]:
        frame["_successes"] = pd.to_numeric(
            frame[contract.primary_numerator], errors="coerce"
        )
        frame["_trials"] = pd.to_numeric(
            frame[contract.primary_denominator], errors="coerce"
        )
    same_objective_peers = same_objective_peers.loc[
        same_objective_peers["_trials"].gt(0)
        & same_objective_peers["_successes"].ge(0)
        & same_objective_peers["_successes"].le(same_objective_peers["_trials"])
    ]
    peers = peers.loc[
        peers["_trials"].gt(0)
        & peers["_successes"].ge(0)
        & peers["_successes"].le(peers["_trials"])
    ]
    item = {
        "Campaign": row["campaign_name"],
        "Own evidence": f"{row['score_successes']:.0f} / {row['score_trials']:.0f}",
        "Same-objective peers": len(same_objective_peers),
        "Selected peers": len(peers),
        "Benchmark source": row["benchmark_scope"],
        "Benchmark quality": row["benchmark_quality"],
        "Peer evidence": (
            f"{peers['_successes'].sum():.0f} / {peers['_trials'].sum():.0f}"
            if len(peers) else "None"
        ),
        "Prior center": np.nan,
        "Prior strength": np.nan,
        "Prior alpha": np.nan,
        "Prior beta": np.nan,
        "Posterior alpha": np.nan,
        "Posterior beta": np.nan,
        "Corrected rate": row["corrected_rate"],
        "Portfolio context": row["portfolio_context_benchmark"],
    }
    if len(peers) >= contract.minimum_peer_entities and row["score_trials"] > 0:
        prior = fit_beta_prior(peers["_successes"], peers["_trials"])
        failures = row["score_trials"] - row["score_successes"]
        item.update({
            "Prior center": prior.mean,
            "Prior strength": prior.strength,
            "Prior alpha": prior.alpha,
            "Prior beta": prior.beta,
            "Posterior alpha": prior.alpha + row["score_successes"],
            "Posterior beta": prior.beta + failures,
        })
    prior_rows.append(item)

prior_table = pd.DataFrame(prior_rows)
display(prior_table.style.format({
    "Prior center": "{:.2%}", "Prior strength": "{:.2f}",
    "Prior alpha": "{:.2f}", "Prior beta": "{:.2f}",
    "Posterior alpha": "{:.2f}", "Posterior beta": "{:.2f}",
    "Corrected rate": "{:.2%}", "Portfolio context": "{:.2%}",
}, na_rep="Insufficient peers"))

,Campaign,Own evidence,Same-objective peers,Selected peers,Benchmark source,Benchmark quality,Peer evidence,Prior center,Prior strength,Prior alpha,Prior beta,Posterior alpha,Posterior beta,Corrected rate,Portfolio context
0,Awareness Boost January,13033 / 1089878,1,2,shared_primary_kpi_group,provisional,29002 / 2338903,1.24%,115518.70,1432.44,114086.27,14465.44,1190931.27,1.20%,1.11%
1,Summer Retention Push,22208 / 1806267,1,2,shared_primary_kpi_group,provisional,19827 / 1622514,1.22%,37985.17,464.19,37520.98,22672.19,1821579.98,1.23%,1.11%


## 6. Raw Rate, Corrected Rate, Benchmark, And Range

The corrected rate is a weighted compromise between peer evidence and the campaign's
own evidence. Small samples move more toward the selected peer center; large samples
retain more of their raw result. The horizontal line is the campaign's 95% plausible
score range. The portfolio-context point is displayed for orientation only.

In [6]:
score_chart_data = campaigns.dropna(subset=["corrected_rate"]).copy()
if score_chart_data.empty:
    display(Markdown(
        "**No corrected campaign scores:** neither the same objective nor a configured "
        "compatible fallback group has two valid peers."
    ))
else:
    y = alt.Y("campaign_name:N", sort="-x", title=None)
    ranges = alt.Chart(score_chart_data).mark_rule(strokeWidth=4, color="#8b95a5").encode(
        y=y,
        x=alt.X("range_low:Q", title="Primary rate", axis=alt.Axis(format=".0%")),
        x2="range_high:Q",
        tooltip=[
            alt.Tooltip("campaign_name:N", title="Campaign"),
            alt.Tooltip("range_low:Q", title="95% low", format=".2%"),
            alt.Tooltip("range_high:Q", title="95% high", format=".2%"),
        ],
    )
    points = score_chart_data[[
        "campaign_name", "raw_rate", "corrected_rate", "benchmark",
        "portfolio_context_benchmark"
    ]].melt("campaign_name", var_name="Estimate", value_name="Rate")
    point_layer = alt.Chart(points).mark_point(filled=True, size=100).encode(
        y=y,
        x=alt.X("Rate:Q", axis=alt.Axis(format=".0%")),
        color=alt.Color(
            "Estimate:N",
            scale=alt.Scale(
                domain=[
                    "raw_rate", "corrected_rate", "benchmark",
                    "portfolio_context_benchmark"
                ],
                range=["#2f6f8f", "#16815d", "#d68c16", "#7b61a8"],
            ),
            legend=alt.Legend(orient="top", title=None),
        ),
        shape=alt.Shape("Estimate:N", legend=None),
        tooltip=[
            alt.Tooltip("campaign_name:N", title="Campaign"),
            alt.Tooltip("Estimate:N"),
            alt.Tooltip("Rate:Q", format=".2%"),
        ],
    )
    display((ranges + point_layer).properties(
        width=760, height=max(140, len(score_chart_data) * 52)
    ))

alt.LayerChart(...)

## 7. Probability Better And The Funding Decision

For each scored campaign, the engine creates 5,000 plausible campaign rates and 5,000
plausible peer rates. `probability_better` is the fraction where favorable lift is
above zero.

```text
higher-is-better lift = campaign draw - benchmark draw
lower-is-better lift  = benchmark draw - campaign draw

SCALE: lift low > 0
KILL:  lift high < 0
HOLD:  lift range crosses 0
```

Probability better supports interpretation and the two-factor budget priority. The complete lift
range, not the probability alone, controls the statistical decision. A provisional
fallback can still calculate a statistical direction, but its final funding action is
always capped at `KEEP_AS_TEST`.

In [7]:
decision_table = campaigns[[
    "campaign_name", "raw_rate", "corrected_rate", "benchmark",
    "benchmark_scope", "benchmark_quality", "portfolio_context_benchmark",
    "expected_lift", "lift_low", "lift_high", "probability_better",
    "statistical_decision", "recommended_action",
]].rename(columns={
    "campaign_name": "Campaign", "raw_rate": "Raw rate",
    "corrected_rate": "Corrected rate", "benchmark": "Benchmark",
    "benchmark_scope": "Benchmark source",
    "benchmark_quality": "Benchmark quality",
    "portfolio_context_benchmark": "Portfolio context",
    "expected_lift": "Expected lift", "lift_low": "Lift low",
    "lift_high": "Lift high", "probability_better": "Probability better",
    "statistical_decision": "Statistical decision",
    "recommended_action": "Final action",
})
display(decision_table.style.format({
    "Raw rate": "{:.2%}", "Corrected rate": "{:.2%}",
    "Benchmark": "{:.2%}", "Expected lift": "{:+.2%}",
    "Lift low": "{:+.2%}", "Lift high": "{:+.2%}",
    "Probability better": "{:.2%}", "Portfolio context": "{:.2%}",
}, na_rep="Not available"))

lift_data = campaigns.dropna(subset=["lift_low", "lift_high"]).copy()
if not lift_data.empty:
    lift_ranges = alt.Chart(lift_data).mark_rule(strokeWidth=5).encode(
        y=alt.Y("campaign_name:N", sort="-x", title=None),
        x=alt.X("lift_low:Q", title="Favorable lift vs peer", axis=alt.Axis(format="+.0%")),
        x2="lift_high:Q",
        color=alt.Color(
            "statistical_decision:N",
            scale=alt.Scale(
                domain=list(DECISION_COLORS), range=list(DECISION_COLORS.values())
            ),
            legend=alt.Legend(orient="top", title="Decision"),
        ),
        tooltip=[
            alt.Tooltip("campaign_name:N", title="Campaign"),
            alt.Tooltip("lift_low:Q", format="+.2%"),
            alt.Tooltip("expected_lift:Q", format="+.2%"),
            alt.Tooltip("lift_high:Q", format="+.2%"),
            alt.Tooltip("probability_better:Q", format=".2%"),
        ],
    )
    expected = alt.Chart(lift_data).mark_point(
        filled=True, color="#20262e", size=90
    ).encode(y=alt.Y("campaign_name:N", sort="-x"), x="expected_lift:Q")
    zero = alt.Chart(pd.DataFrame({"zero": [0]})).mark_rule(
        strokeDash=[5, 4], color="#20262e"
    ).encode(x="zero:Q")
    display((lift_ranges + expected + zero).properties(
        width=760, height=max(140, len(lift_data) * 52)
    ))

,Campaign,Raw rate,Corrected rate,Benchmark,Benchmark source,Benchmark quality,Portfolio context,Expected lift,Lift low,Lift high,Probability better,Statistical decision,Final action
1,Awareness Boost January,1.20%,1.20%,1.24%,shared_primary_kpi_group,provisional,1.11%,-0.04%,-0.11%,+0.03%,12.24%,hold,keep_as_test
11,Summer Retention Push,1.23%,1.23%,1.22%,shared_primary_kpi_group,provisional,1.11%,+0.01%,-0.10%,+0.12%,55.02%,hold,keep_as_test


alt.LayerChart(...)

## 8. Efficiency Is A Separate Gate

The primary score answers whether the objective outcome was achieved. Efficiency asks
what it cost, or what revenue return it produced. The peer benchmark is the median
efficiency of other campaigns with this objective.

The Awareness-plus-Engagement fallback does **not** combine efficiency metrics:
Awareness keeps CPM, while Engagement keeps CPC. Their shared Link CTR is useful for
stabilizing response evidence, but their costs answer different business questions.

A statistical `SCALE` is promoted to final `SCALE` only when efficiency is at least as
good as its peer benchmark. Efficiency is not mixed into the primary score.

In [8]:
efficiency_table = campaigns[[
    "campaign_name", "efficiency_metric", "efficiency_direction",
    "efficiency_value", "efficiency_benchmark", "efficiency_peer_count",
    "efficiency_comparison",
]].rename(columns={
    "campaign_name": "Campaign", "efficiency_metric": "Metric",
    "efficiency_direction": "Better direction", "efficiency_value": "Value",
    "efficiency_benchmark": "Peer median", "efficiency_peer_count": "Peers",
    "efficiency_comparison": "Comparison",
})
display(efficiency_table.style.format({
    "Value": "{:,.3f}", "Peer median": "{:,.3f}"
}, na_rep="Not available"))

efficiency_long = efficiency_table.melt(
    id_vars=["Campaign", "Metric", "Better direction", "Comparison"],
    value_vars=["Value", "Peer median"],
    var_name="Measure", value_name="Efficiency",
).dropna(subset=["Efficiency"])
if not efficiency_long.empty:
    efficiency_chart = alt.Chart(efficiency_long).mark_bar().encode(
        y=alt.Y("Campaign:N", title=None),
        x=alt.X("Efficiency:Q", title=contract.efficiency_metric.replace("_", " ").title()),
        yOffset="Measure:N",
        color=alt.Color(
            "Measure:N",
            scale=alt.Scale(domain=["Value", "Peer median"], range=["#2f6f8f", "#d68c16"]),
            legend=alt.Legend(orient="top", title=None),
        ),
        tooltip=["Campaign", "Metric", "Better direction", "Measure", "Efficiency"],
    ).properties(width=760, height=max(140, len(campaigns) * 58))
    display(efficiency_chart)

,Campaign,Metric,Better direction,Value,Peer median,Peers,Comparison
1,Awareness Boost January,cpm,lower,5.475,4.986,1,worse_than_peer
11,Summer Retention Push,cpm,lower,4.986,5.475,1,better_than_peer


alt.Chart(...)

## 9. Final Campaign Action And Budget

The statistical decision and evidence status create reporting labels:

| Statistical evidence | Efficiency | Final action |
|---|---|---|
| SCALE | Better or equal to peer | SCALE |
| SCALE | Worse or unavailable | KEEP_AS_TEST |
| HOLD | Any | KEEP_AS_TEST |
| KILL | Any | DO_NOT_FUND |
| Too little evidence | Any | INSUFFICIENT_EVIDENCE |

For a `provisional` shared-Link-CTR benchmark, any assessable statistical result is
capped at `KEEP_AS_TEST`. It cannot trigger `SCALE` or `DO_NOT_FUND`.

For this POC, those labels do not gate budget. Previous-cycle spend shares preserve
each objective envelope, then all campaigns with both components share the full
envelope using `probability_better x semantic range_low`. Missing components receive
zero; an objective with no valid priorities remains unallocated.

In [9]:
recommendation_rows = []
for _, row in campaigns.iterrows():
    allocation = allocations[str(row["campaign_id"])]
    recommendation_rows.append({
        "Campaign": row["campaign_name"],
        "Benchmark source": row["benchmark_scope"],
        "Benchmark quality": row["benchmark_quality"],
        "Statistical decision": row["statistical_decision"],
        "Efficiency": row["efficiency_comparison"],
        "Final action": row["recommended_action"],
        "Budget pool": allocation.budget_pool,
        "Primary probability": allocation.primary_probability_component,
        "Quality lower bound": allocation.semantic_priority,
        "Priority product": allocation.allocation_priority,
        "Normalized weight": allocation.allocation_weight,
        "Budget units": allocation.recommended_budget_units,
        "Previous-spend scenario": allocation.previous_spend_budget_units,
        "Allocation basis": allocation.allocation_basis,
        "Portfolio share": allocation.recommended_budget_share,
        "Reason codes": ", ".join(allocation.reason_codes),
    })
recommendation_table = pd.DataFrame(recommendation_rows)
display(recommendation_table.style.format({
    "Budget units": "{:.2f}", "Portfolio share": "{:.2%}",
    "Primary probability": "{:.2%}", "Quality lower bound": "{:.2%}",
    "Priority product": "{:.4f}", "Normalized weight": "{:.2%}"
}))

budget_data = recommendation_table[recommendation_table["Budget units"].gt(0)]
if not budget_data.empty:
    budget_chart = alt.Chart(budget_data).mark_bar().encode(
        y=alt.Y("Campaign:N", sort="-x", title=None),
        x=alt.X("Budget units:Q", title="Recommended next-cycle budget units"),
        color=alt.value("#16815d"),
        tooltip=["Campaign", "Final action", "Primary probability",
                 "Quality lower bound", "Priority product",
                 alt.Tooltip("Budget units:Q", format=".2f")],
    ).properties(width=760, height=max(130, len(budget_data) * 48))
    display(budget_chart)
else:
    display(Markdown("**No budget is assigned to this objective under the current evidence rules.**"))

,Campaign,Benchmark source,Benchmark quality,Statistical decision,Efficiency,Final action,Budget pool,Primary probability,Quality lower bound,Priority product,Normalized weight,Budget units,Previous-spend scenario,Allocation basis,Portfolio share,Reason codes
0,Awareness Boost January,shared_primary_kpi_group,provisional,hold,worse_than_peer,keep_as_test,score_based,12.24%,12.63%,0.0155,32.05%,1.19,1.483230,primary_probability_x_semantic_lower_bound,1.19%,"PRIMARY_LIFT_RANGE_CROSSES_ZERO, SAME_OBJECTIVE_PEERS_INSUFFICIENT, FALLBACK_TO_SHARED_PRIMARY_KPI_GROUP, FALLBACK_ACTION_CAPPED_AT_TEST, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND"
1,Summer Retention Push,shared_primary_kpi_group,provisional,hold,better_than_peer,keep_as_test,score_based,55.02%,5.96%,0.0328,67.95%,2.53,2.238931,primary_probability_x_semantic_lower_bound,2.53%,"PRIMARY_LIFT_RANGE_CROSSES_ZERO, SAME_OBJECTIVE_PEERS_INSUFFICIENT, FALLBACK_TO_SHARED_PRIMARY_KPI_GROUP, FALLBACK_ACTION_CAPPED_AT_TEST, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND"


alt.Chart(...)

## 10. Named Tests For KEEP_AS_TEST Campaigns

The score-based budget is independent of action labels. When a funded campaign is
labelled `KEEP_AS_TEST`, the report also creates a specific hypothesis with a success
rule, failure rule, and stop rule for learning during the next cycle.

In [10]:
if objective_tests:
    tests_table = pd.DataFrame([
        {
            "Campaign": item.campaign_name,
            "Budget units": item.assigned_budget_units,
            "Hypothesis": item.hypothesis,
            "Primary metric": item.primary_metric,
            "Success rule": item.success_rule,
            "Failure rule": item.failure_rule,
            "Stop rule": item.stop_rule,
        }
        for item in objective_tests
    ])
    display(tests_table.style.format({"Budget units": "{:.2f}"}))
else:
    display(Markdown(
        "No funded KEEP_AS_TEST campaign is present for this objective."
    ))

,Campaign,Budget units,Hypothesis,Primary metric,Success rule,Failure rule,Stop rule
0,Awareness Boost January,1.19,"Clarify the offer and address the observed product_fit barrier in a named variant against the current message. Hypothesis: this increases ad alignment and improves link ctr in the next cycle. The observed association is a hypothesis, not an established cause.",link_ctr,"At cycle end, check the primary favorable-lift range, efficiency, and the separate semantic range. A provisional Link CTR benchmark remains capped at KEEP_AS_TEST.","At cycle end, DO_NOT_FUND requires a decision-grade primary lift upper bound below zero with sufficient evidence. Semantic quality alone does not determine the final action.","Stop when the assigned budget is spent or the next cycle ends, whichever comes first; wait for outcomes to mature before scoring."
1,Summer Retention Push,2.53,"Clarify the offer and address the observed product_fit barrier in a named variant against the current message. Hypothesis: this increases ad alignment and improves link ctr in the next cycle. The observed association is a hypothesis, not an established cause.",link_ctr,"At cycle end, check the primary favorable-lift range, efficiency, and the separate semantic range. A provisional Link CTR benchmark remains capped at KEEP_AS_TEST.","At cycle end, DO_NOT_FUND requires a decision-grade primary lift upper bound below zero with sufficient evidence. Semantic quality alone does not determine the final action.","Stop when the assigned budget is spent or the next cycle ends, whichever comes first; wait for outcomes to mature before scoring."


## 11. Adset, Ad, Creative, And Audience Evidence

The same objective KPI and benchmark hierarchy are applied at every level.
Campaign and adset are normal budget-control layers. Ads are run/test/stop candidates.
Creative and audience views identify reusable patterns; their budgets are not added
separately.

A larger corrected score is a leader to investigate, not automatically a winner. The
range-based decision and final action remain authoritative.

In [11]:
child_rows = []
for level in ["adset", "ad", "creative", "audience"]:
    frame = result.scorecards[level].loc[
        result.scorecards[level]["objective"].eq(FOCUS_OBJECTIVE)
    ].copy()
    for _, row in frame.iterrows():
        child_rows.append({
            "Level": level,
            "Campaign": row["campaign_name"],
            "Entity": row["entity_name"],
            "Successes": row["score_successes"],
            "Trials": row["score_trials"],
            "Raw rate": row["raw_rate"],
            "Corrected rate": row["corrected_rate"],
            "Lift low": row["lift_low"],
            "Lift high": row["lift_high"],
            "Probability better": row["probability_better"],
            "Benchmark source": row["benchmark_scope"],
            "Benchmark quality": row["benchmark_quality"],
            "Statistical decision": row["statistical_decision"],
            "Final action": row["recommended_action"],
            "Spend": row["spend"],
        })
child_summary = pd.DataFrame(child_rows)
display(pd.crosstab(
    [child_summary["Level"]], child_summary["Final action"], margins=True
))

for level in ["adset", "ad", "creative", "audience"]:
    subset = child_summary[child_summary["Level"].eq(level)].copy()
    subset = subset.sort_values(
        ["Probability better", "Trials"], ascending=[False, False], na_position="last"
    ).head(12)
    display(Markdown(f"### {level.title()} leaders and test candidates"))
    display(subset.style.format({
        "Raw rate": "{:.2%}", "Corrected rate": "{:.2%}",
        "Lift low": "{:+.2%}", "Lift high": "{:+.2%}",
        "Probability better": "{:.2%}", "Spend": "{:,.2f}",
    }, na_rep="Not available"))

Final action,keep_as_test,All
Level,,
ad,3,3
adset,3,3
audience,3,3
creative,2,2
All,11,11


### Adset leaders and test candidates

,Level,Campaign,Entity,Successes,Trials,Raw rate,Corrected rate,Lift low,Lift high,Probability better,Benchmark source,Benchmark quality,Statistical decision,Final action,Spend
1,adset,Summer Retention Push,Existing customer Custom Audience retention,11080.000000,842175.000000,1.32%,1.30%,+0.06%,+0.18%,100.00%,same_objective,decision_grade,scale,keep_as_test,"4,708.10"
0,adset,Awareness Boost January,Cairo+Alex broad awareness,13033.000000,1089878.000000,1.20%,1.20%,-0.27%,+0.18%,38.86%,same_objective,decision_grade,hold,keep_as_test,"5,966.66"
2,adset,Summer Retention Push,Lookalike recent buyers warm retention,11128.000000,964092.000000,1.15%,1.16%,-0.26%,+0.07%,12.86%,same_objective,decision_grade,hold,keep_as_test,"4,298.55"


### Ad leaders and test candidates

,Level,Campaign,Entity,Successes,Trials,Raw rate,Corrected rate,Lift low,Lift high,Probability better,Benchmark source,Benchmark quality,Statistical decision,Final action,Spend
4,ad,Summer Retention Push,Retention Custom Audience warm,11080.000000,842175.000000,1.32%,1.30%,+0.06%,+0.18%,100.00%,same_objective,decision_grade,scale,keep_as_test,"4,708.10"
3,ad,Awareness Boost January,Awareness creative Cairo+Alex,13033.000000,1089878.000000,1.20%,1.20%,-0.27%,+0.18%,40.34%,same_objective,decision_grade,hold,keep_as_test,"5,966.66"
5,ad,Summer Retention Push,Retention LAL warm recent,11128.000000,964092.000000,1.15%,1.16%,-0.26%,+0.07%,14.40%,same_objective,decision_grade,hold,keep_as_test,"4,298.55"


### Creative leaders and test candidates

,Level,Campaign,Entity,Successes,Trials,Raw rate,Corrected rate,Lift low,Lift high,Probability better,Benchmark source,Benchmark quality,Statistical decision,Final action,Spend
7,creative,Summer Retention Push,Summer retention awareness warm,22208.000000,1806267.000000,1.23%,1.23%,-0.11%,+0.12%,55.20%,shared_primary_kpi_group,provisional,hold,keep_as_test,"9,006.65"
6,creative,Awareness Boost January,Always-on premium acquisition default,13033.000000,1089878.000000,1.20%,1.20%,-0.11%,+0.02%,11.00%,shared_primary_kpi_group,provisional,hold,keep_as_test,"5,966.66"


### Audience leaders and test candidates

,Level,Campaign,Entity,Successes,Trials,Raw rate,Corrected rate,Lift low,Lift high,Probability better,Benchmark source,Benchmark quality,Statistical decision,Final action,Spend
9,audience,Summer Retention Push,Custom,11080.000000,842175.000000,1.32%,1.30%,+0.06%,+0.18%,99.98%,same_objective,decision_grade,scale,keep_as_test,"4,708.10"
8,audience,Awareness Boost January,Broad,13033.000000,1089878.000000,1.20%,1.20%,-0.27%,+0.19%,39.68%,same_objective,decision_grade,hold,keep_as_test,"5,966.66"
10,audience,Summer Retention Push,Lookalike,11128.000000,964092.000000,1.15%,1.16%,-0.27%,+0.07%,13.00%,same_objective,decision_grade,hold,keep_as_test,"4,298.55"


In [12]:
child_chart_data = child_summary.dropna(subset=["Lift low", "Lift high"]).copy()
child_chart_data = child_chart_data.sort_values(
    "Probability better", ascending=False
).groupby("Level", as_index=False).head(10)
for level in ["adset", "ad", "creative", "audience"]:
    level_data = child_chart_data[child_chart_data["Level"].eq(level)]
    if level_data.empty:
        continue
    base = alt.Chart(level_data)
    child_chart = base.mark_rule(strokeWidth=4).encode(
        y=alt.Y("Entity:N", sort="-x", title=None),
        x=alt.X("Lift low:Q", title="Favorable lift vs selected peer", axis=alt.Axis(format="+.0%")),
        x2="Lift high:Q",
        color=alt.Color(
            "Statistical decision:N",
            scale=alt.Scale(domain=list(DECISION_COLORS), range=list(DECISION_COLORS.values())),
            legend=alt.Legend(orient="top"),
        ),
        tooltip=[
            "Level", "Campaign", "Entity", "Successes", "Trials",
            alt.Tooltip("Probability better:Q", format=".2%"),
            "Final action",
        ],
    )
    zero = base.mark_rule(color="#20262e", strokeDash=[5, 4]).encode(
        x=alt.datum(0)
    )
    display(Markdown(f"### {level.title()} favorable-lift ranges"))
    display((child_chart + zero).properties(
        width=760, height=max(180, len(level_data) * 38)
    ))

### Adset favorable-lift ranges

alt.LayerChart(...)

### Ad favorable-lift ranges

alt.LayerChart(...)

### Creative favorable-lift ranges

alt.LayerChart(...)

### Audience favorable-lift ranges

alt.LayerChart(...)

## 12. What Conversation Signals Contribute

Use message alignment, customer needs, and value drivers to learn whether the attention attracted by the ad matches its promise. Purchase intent is downstream diagnostic evidence, not the awareness score.

The semantic artifact is joined back to structured outcomes only after extraction.
The extraction model did not receive revenue, outcome, customer identity, budget, or
funding decisions. `unknown` is excluded from assessable denominators rather than
converted to false.

`next_step_order_progression_rate` means an agreed next step followed by an observed
structured order. It is an order-progression proxy, not proof that every promised task
was completed.

The following sections calculate a separate objective-specific quality score. These
full-conversation labels describe historical interest and agreement; they may record
an agreement followed by cancellation. They are not predictions of later purchases.

In [13]:
diagnostic_columns = [
    "semantic_coverage_rate", "high_purchase_intent_rate",
    "price_blocking_rate", "barrier_resolution_rate",
    "ad_alignment_rate", "agent_helpful_rate",
    "next_step_agreement_rate", "next_step_order_progression_rate",
]
semantic_table = campaigns[[
    "campaign_name", "semantic_conversations", *diagnostic_columns,
    "top_customer_need", "top_barrier", "top_value_driver",
]].rename(columns={"campaign_name": "Campaign"})
display(semantic_table.style.format(
    {column: "{:.2%}" for column in diagnostic_columns},
    na_rep="Unknown / not assessable",
))

heatmap = semantic_table[["Campaign", *diagnostic_columns]].melt(
    "Campaign", var_name="Signal", value_name="Rate"
).dropna(subset=["Rate"])
if not heatmap.empty:
    base = alt.Chart(heatmap).encode(
        x=alt.X("Signal:N", title=None, axis=alt.Axis(labelAngle=-35)),
        y=alt.Y("Campaign:N", title=None),
    )
    rectangles = base.mark_rect().encode(
        color=alt.Color(
            "Rate:Q", scale=alt.Scale(domain=[0, 0.5, 1], range=["#c44536", "#f2cf63", "#16815d"]),
            legend=alt.Legend(format=".0%", orient="top"),
        ),
        tooltip=["Campaign", "Signal", alt.Tooltip("Rate:Q", format=".2%")],
    )
    labels = base.mark_text(size=11).encode(
        text=alt.Text("Rate:Q", format=".0%"),
        color=alt.condition("datum.Rate < 0.25 || datum.Rate > 0.78", alt.value("white"), alt.value("#20262e")),
    )
    display((rectangles + labels).properties(width=760, height=max(110, len(campaigns) * 48)))

,Campaign,semantic_conversations,semantic_coverage_rate,high_purchase_intent_rate,price_blocking_rate,barrier_resolution_rate,ad_alignment_rate,agent_helpful_rate,next_step_agreement_rate,next_step_order_progression_rate,top_customer_need,top_barrier,top_value_driver
1,Awareness Boost January,4,100.00%,75.00%,0.00%,50.00%,50.00%,100.00%,75.00%,66.67%,شراء Madagascar vanilla beans و sage tea,product_fit,quality
11,Summer Retention Push,16,100.00%,71.43%,0.00%,40.00%,18.75%,100.00%,78.57%,90.91%,"Place an order for groceries: 1L olive oil, sourdough, and parmesan if available.",product_fit,product_fit


alt.LayerChart(...)

### 12.1 Define Conversation Quality Before Counting

Each objective has one explicit rule. These definitions are MVP assumptions that
need transcript review. They do not combine labels using subjective weights.
A positive evidence-bearing label must cite message indexes. All required components
must be known; an unknown component leaves the combined result unknown.

| Objective | Success definition |
|---|---|
| Awareness | Central need aligned with the ad; partial/mismatch are negative |
| Engagement | Commercial inquiry, medium/high specificity, consideration/checkout, aligned/partial ad match |
| Leads | Commercial inquiry, medium/high specificity and medium/high purchase intent |
| Sales | Commercial inquiry, complete sales agreement and accepted next step |

Commercial inquiry means purchase, product information, promotion information or
delivery information. These semantic rates describe customers who chatted. They do
not measure awareness lift or engagement among everyone who viewed an ad.

In [14]:
display(Markdown(f"**This objective:** {contract.semantic.metric}: {contract.semantic.definition}"))
objective_audit = result.semantic_evidence.loc[
    result.semantic_evidence["objective"].eq(FOCUS_OBJECTIVE)
].copy()
display(objective_audit.groupby(["entity_level", "exclusion_reason"], dropna=False).size().rename("Records").reset_index())

**This objective:** ad_alignment: The customer's central need is aligned with the ad promise; partial and mismatch are assessed negatives.

,entity_level,exclusion_reason,Records
0,ad,unresolved,1
1,ad,NaN,19
2,adset,unresolved,1
3,adset,NaN,19
4,audience,unresolved,1
5,audience,NaN,19
6,campaign,unresolved,1
7,campaign,NaN,19
8,creative,unresolved,1
9,creative,NaN,19


### 12.2 Inspect Selection And Raw Evidence

Choose the earliest mature conversation per customer within each entity, using
started_at then conversation ID (missing timestamps last). Choose before checking
labels. Later records do not replace an earlier unknown or missing result.
This prevents repeat chats from multiplying a customer's weight. Open outcomes
stay out of this completed-cycle score. A customer can still appear across entities.

The local audit below connects every selection and exclusion to a conversation ID.
No transcript or customer identity is passed to the recommendation model.

In [15]:
campaign_audit = objective_audit[objective_audit["entity_level"].eq("campaign")].copy()
display(campaign_audit[[
    "campaign_id", "conversation_id", "metric", "selected",
    "exclusion_reason", "signal_available", "success"
]])
semantic_tables = {}
for level, frame in result.scorecards.items():
    focused = frame[frame["objective"].eq(FOCUS_OBJECTIVE)].reset_index(drop=True)
    semantic_tables[level] = pd.concat([
        focused[["entity_id", "entity_name", "campaign_name"]],
        pd.json_normalize(focused["semantic_score"])
    ], axis=1)
quality = semantic_tables["campaign"]
display(quality[[
    "entity_name", "eligible_customers", "successes", "trials",
    "unknown_customers", "missing_customers", "raw_rate", "evidence_status"
]].style.format({"raw_rate": "{:.2%}"}, na_rep="Unknown"))
assert (quality["trials"] + quality["unknown_customers"] + quality["missing_customers"]).equals(quality["eligible_customers"])

,campaign_id,conversation_id,metric,selected,exclusion_reason,signal_available,success
31,120209876543220002,conv_779,ad_alignment,True,None,True,False
45,120209876543220002,conv_073,ad_alignment,True,None,True,True
62,120209876543220002,conv_010,ad_alignment,True,None,True,False
80,120209876543220002,conv_611,ad_alignment,True,None,True,True
532,120209876543220012,conv_604,ad_alignment,True,None,True,False
539,120209876543220012,conv_245,ad_alignment,True,None,True,False
550,120209876543220012,conv_711,ad_alignment,True,None,True,False
552,120209876543220012,conv_230,ad_alignment,True,None,True,False
558,120209876543220012,conv_546,ad_alignment,True,None,True,False
559,120209876543220012,conv_558,ad_alignment,True,None,True,True


,entity_name,eligible_customers,successes,trials,unknown_customers,missing_customers,raw_rate,evidence_status
0,Awareness Boost January,4,2,4,0,0,50.00%,limited
1,Summer Retention Push,15,3,15,0,0,20.00%,sufficient


### 12.3 Calculate The Corrected Quality And Range

With two valid same-objective peers, learn the prior with the same Empirical-Bayes
method used earlier. Exclude the entity itself. Semantic metrics differ across
objectives, so the shared Link CTR fallback does not apply here.

With fewer peers, use Beta(0.5, 0.5), a weak Jeffreys prior: half a success and half a
failure worth of mathematical smoothing, not real customer records. Show its range
but no peer benchmark, lift or probability-better claim.

```text
raw rate = successes / assessable customers
posterior alpha = prior alpha + successes
posterior beta = prior beta + assessable customers - successes
corrected rate = posterior alpha / (posterior alpha + posterior beta)
95% credible range = 2.5th and 97.5th percentiles of posterior draws
```

The code uses 5,000 repeatable draws. With a valid peer prior, favorable lift above
zero across the whole range means supportive; below zero means concerning; crossing
zero means neutral. Fewer than 10 assessable customers means insufficient evidence.
Ten is an MVP evidence floor, not a universal statistical guarantee.

In [16]:
for level, table in semantic_tables.items():
    display(Markdown(f"**{level.title()}: raw evidence through posterior**"))
    display(table[[
        "entity_name", "metric", "successes", "trials", "prior_source", "peer_count",
        "prior_alpha", "prior_beta", "posterior_alpha", "posterior_beta",
        "raw_rate", "corrected_rate", "range_low", "range_high",
        "benchmark", "lift_low", "lift_high", "probability_better", "status"
    ]].style.format({
        **{c: "{:.2%}" for c in ["raw_rate", "corrected_rate", "range_low", "range_high", "benchmark", "probability_better"]},
        **{c: "{:.3f}" for c in ["prior_alpha", "prior_beta", "posterior_alpha", "posterior_beta"]},
        "lift_low": "{:+.2%}", "lift_high": "{:+.2%}"
    }, na_rep="Not available"))
    available = table[table["trials"].gt(0)]
    calculated = available["posterior_alpha"] / (available["posterior_alpha"] + available["posterior_beta"])
    assert np.allclose(calculated, available["corrected_rate"])

**Campaign: raw evidence through posterior**

,entity_name,metric,successes,trials,prior_source,peer_count,prior_alpha,prior_beta,posterior_alpha,posterior_beta,raw_rate,corrected_rate,range_low,range_high,benchmark,lift_low,lift_high,probability_better,status
0,Awareness Boost January,ad_alignment,2,4,jeffreys_no_peer_comparison,1,0.500,0.500,2.500,2.500,50.00%,50.00%,12.63%,87.71%,Not available,Not available,Not available,Not available,insufficient_evidence
1,Summer Retention Push,ad_alignment,3,15,jeffreys_no_peer_comparison,1,0.500,0.500,3.500,12.500,20.00%,21.88%,5.96%,43.29%,Not available,Not available,Not available,Not available,no_peer_comparison


**Adset: raw evidence through posterior**

,entity_name,metric,successes,trials,prior_source,peer_count,prior_alpha,prior_beta,posterior_alpha,posterior_beta,raw_rate,corrected_rate,range_low,range_high,benchmark,lift_low,lift_high,probability_better,status
0,Cairo+Alex broad awareness,ad_alignment,2,4,same_objective_empirical,2,0.612,2.186,2.612,4.186,50.00%,38.42%,8.89%,73.77%,21.88%,-44.86%,+64.02%,75.48%,insufficient_evidence
1,Existing customer Custom Audience retention,ad_alignment,1,10,same_objective_empirical,2,2.025,2.475,3.025,11.475,10.00%,20.86%,5.10%,44.55%,45.00%,-70.09%,+18.25%,16.52%,neutral
2,Lookalike recent buyers warm retention,ad_alignment,2,5,same_objective_empirical,2,0.288,0.948,2.288,3.948,40.00%,36.70%,7.38%,74.77%,23.33%,-63.06%,+66.52%,71.78%,insufficient_evidence


**Ad: raw evidence through posterior**

,entity_name,metric,successes,trials,prior_source,peer_count,prior_alpha,prior_beta,posterior_alpha,posterior_beta,raw_rate,corrected_rate,range_low,range_high,benchmark,lift_low,lift_high,probability_better,status
0,Awareness creative Cairo+Alex,ad_alignment,2,4,same_objective_empirical,2,0.612,2.186,2.612,4.186,50.00%,38.42%,8.75%,74.41%,21.88%,-45.34%,+64.59%,74.56%,insufficient_evidence
1,Retention Custom Audience warm,ad_alignment,1,10,same_objective_empirical,2,2.025,2.475,3.025,11.475,10.00%,20.86%,5.05%,43.89%,45.00%,-71.17%,+19.33%,16.36%,neutral
2,Retention LAL warm recent,ad_alignment,2,5,same_objective_empirical,2,0.288,0.948,2.288,3.948,40.00%,36.70%,7.11%,73.25%,23.33%,-62.43%,+64.77%,72.58%,insufficient_evidence


**Creative: raw evidence through posterior**

,entity_name,metric,successes,trials,prior_source,peer_count,prior_alpha,prior_beta,posterior_alpha,posterior_beta,raw_rate,corrected_rate,range_low,range_high,benchmark,lift_low,lift_high,probability_better,status
0,Always-on premium acquisition default,ad_alignment,2,4,jeffreys_no_peer_comparison,1,0.500,0.500,2.500,2.500,50.00%,50.00%,12.22%,87.88%,Not available,Not available,Not available,Not available,insufficient_evidence
1,Summer retention awareness warm,ad_alignment,3,15,jeffreys_no_peer_comparison,1,0.500,0.500,3.500,12.500,20.00%,21.88%,5.91%,43.45%,Not available,Not available,Not available,Not available,no_peer_comparison


**Audience: raw evidence through posterior**

,entity_name,metric,successes,trials,prior_source,peer_count,prior_alpha,prior_beta,posterior_alpha,posterior_beta,raw_rate,corrected_rate,range_low,range_high,benchmark,lift_low,lift_high,probability_better,status
0,Broad,ad_alignment,2,4,same_objective_empirical,2,0.612,2.186,2.612,4.186,50.00%,38.42%,8.58%,73.90%,21.88%,-44.28%,+64.40%,75.14%,insufficient_evidence
1,Custom,ad_alignment,1,10,same_objective_empirical,2,2.025,2.475,3.025,11.475,10.00%,20.86%,4.89%,43.49%,45.00%,-68.96%,+19.44%,16.50%,neutral
2,Lookalike,ad_alignment,2,5,same_objective_empirical,2,0.288,0.948,2.288,3.948,40.00%,36.70%,7.14%,74.14%,23.33%,-64.01%,+66.76%,71.20%,insufficient_evidence


### 12.4 Compare Quality Ranges

Read the point together with the line. A high point with a wide range has limited
evidence. These ranges assume the extracted labels are correct; they do not account
for systematic LLM mistakes. Overlapping ranges should not be presented as proven
winners. Primary outcome scores remain separate because their denominators and
business meanings differ.

In [17]:
plotted = quality.dropna(subset=["corrected_rate"])
if not plotted.empty:
    base = alt.Chart(plotted).encode(y=alt.Y("entity_name:N", title=None))
    bounds = base.mark_rule(strokeWidth=4, color="#218380").encode(
        x=alt.X("range_low:Q", title="Conversation quality", axis=alt.Axis(format=".0%"), scale=alt.Scale(domain=[0, 1])),
        x2="range_high:Q",
        tooltip=["entity_name", "successes", "trials", "prior_source", "status"]
    )
    points = base.mark_point(filled=True, color="#20262e", size=90).encode(x="corrected_rate:Q")
    display((bounds + points).properties(width=760, height=max(150, 45 * len(plotted))))

alt.LayerChart(...)

### 12.5 Check Whether Quality Is Associated With Outcomes

The next table compares orders and deliveries between semantic-positive,
semantic-negative and unknown selected conversations. It is a retrospective
association, not causal evidence or an independent predictive evaluation: the
extractor saw the full conversation, including historical checkout steps.
This is a starting point for manually reviewing whether the definitions are useful.
Small groups and missing labels can make the differences unstable.

In [18]:
selected = campaign_audit[campaign_audit["selected"]].copy()
selected["Semantic result"] = selected["success"].map({True: "positive", False: "negative"}).fillna("unknown")
associations = selected.groupby(["campaign_id", "Semantic result"]).agg(
    customers=("conversation_id", "size"), orders=("has_order", "sum"), deliveries=("is_delivered", "sum")
).reset_index()
associations["order_rate"] = associations["orders"] / associations["customers"]
associations["delivery_rate"] = associations["deliveries"] / associations["customers"]
display(associations.style.format({"order_rate": "{:.2%}", "delivery_rate": "{:.2%}"}))

,campaign_id,Semantic result,customers,orders,deliveries,order_rate,delivery_rate
0,120209876543220002,negative,2,1,0,50.00%,0.00%
1,120209876543220002,positive,2,1,1,50.00%,50.00%
2,120209876543220012,negative,12,8,7,66.67%,58.33%
3,120209876543220012,positive,3,1,0,33.33%,0.00%


### 12.6 How Primary And Conversation Evidence Allocate Budget

Every campaign with both required components receives a share of its objective's
full envelope:

```text
priority = primary probability_better x semantic 95% lower bound
weight = priority / sum of campaign priorities in the objective
campaign budget = full objective envelope x weight
```

Objective envelopes retain previous-cycle spend shares because their primary and
semantic metrics are not comparable across objectives. Within an objective, action
labels do not gate this POC allocation. Missing either component gives the campaign
zero allocation. The priority product is a policy index, not a joint probability or
a learned return-on-spend optimum.

Review labels against transcripts before operational use. No manual validation is
claimed here. The table shows exactly how much this assumption changes the budget.

In [19]:
impact = pd.DataFrame([
    {"Campaign": item.campaign_name,
     "Primary probability": item.primary_probability_component,
     "Quality lower bound": item.semantic_priority,
     "Priority product": item.allocation_priority,
     "Normalized weight": item.allocation_weight,
     "Basis": item.allocation_basis,
     "Previous-spend scenario": item.previous_spend_budget_units,
     "Two-factor scenario": item.recommended_budget_units,
     "Change": item.recommended_budget_units - item.previous_spend_budget_units}
    for item in result.allocations if item.objective == FOCUS_OBJECTIVE
])
display(impact.style.format({
    "Primary probability": "{:.2%}", "Quality lower bound": "{:.2%}",
    "Priority product": "{:.4f}", "Normalized weight": "{:.2%}",
    "Previous-spend scenario": "{:.3f}", "Two-factor scenario": "{:.3f}",
    "Change": "{:+.3f}"
}, na_rep="Not available"))

,Campaign,Primary probability,Quality lower bound,Priority product,Normalized weight,Basis,Previous-spend scenario,Two-factor scenario,Change
0,Awareness Boost January,12.24%,12.63%,0.0155,32.05%,primary_probability_x_semantic_lower_bound,1.483,1.193,-0.290
1,Summer Retention Push,55.02%,5.96%,0.0328,67.95%,primary_probability_x_semantic_lower_bound,2.239,2.529,+0.290


## 13. Campaign-By-Campaign Recommendation

The statements below translate the locked deterministic outputs. They do not create a
new decision from the semantic signals.

In [20]:
sections = []
for _, row in campaigns.sort_values("spend", ascending=False).iterrows():
    allocation = allocations[str(row["campaign_id"])]
    probability = (
        f"{row['probability_better']:.1%}"
        if pd.notna(row["probability_better"])
        else "not available"
    )
    lift = (
        f"[{row['lift_low']:+.1%}, {row['lift_high']:+.1%}]"
        if pd.notna(row["lift_low"]) else "not available"
    )
    portfolio_context = (
        f"{row['portfolio_context_benchmark']:.1%}"
        if pd.notna(row["portfolio_context_benchmark"])
        else "not available"
    )
    semantic_sentence = (
        f"Semantic coverage is {row['semantic_coverage_rate']:.1%}; the leading "
        f"barrier is {row['top_barrier'] or 'not established'} and the leading "
        f"value driver is {row['top_value_driver'] or 'not established'}."
        if pd.notna(row["semantic_coverage_rate"])
        else "Conversation semantics are not yet available for this campaign."
    )
    sections.append(
        f"### {row['campaign_name']}\n"
        f"- **Action:** `{row['recommended_action'].upper()}`; "
        f"statistical decision `{row['statistical_decision'].upper()}`.\n"
        f"- **Evidence:** {row['score_successes']:.0f} successes from "
        f"{row['score_trials']:.0f} eligible trials; raw {row['raw_rate']:.1%}.\n"
        f"- **Uncertainty:** probability better {probability}; lift range {lift}.\n"
        f"- **Benchmark:** \`{row['benchmark_scope']}\` with "
        f"\`{row['benchmark_quality']}\` quality; portfolio context "
        f"{portfolio_context} is descriptive only.\n"
        f"- **Efficiency:** {row['efficiency_metric'].replace('_', ' ')} is "
        f"`{row['efficiency_comparison']}`.\n"
        f"- **Budget:** {allocation.recommended_budget_units:.2f} "
        f"{result.recommendation_input.cycle.currency} units; priority is "
        f"{allocation.primary_probability_component:.1%} x "
        f"{allocation.semantic_priority:.1%}.\n"
        f"- **Conversation context:** {semantic_sentence}\n"
        f"- **Reason codes:** {', '.join(allocation.reason_codes)}."
    )
display(Markdown("\n\n".join(sections)))

### Summer Retention Push
- **Action:** `KEEP_AS_TEST`; statistical decision `HOLD`.
- **Evidence:** 22208 successes from 1806267 eligible trials; raw 1.2%.
- **Uncertainty:** probability better 55.0%; lift range [-0.1%, +0.1%].
- **Benchmark:** \`shared_primary_kpi_group\` with \`provisional\` quality; portfolio context 1.1% is descriptive only.
- **Efficiency:** cpm is `better_than_peer`.
- **Budget:** 2.53 units units; priority is 55.0% x 6.0%.
- **Conversation context:** Semantic coverage is 100.0%; the leading barrier is product_fit and the leading value driver is product_fit.
- **Reason codes:** PRIMARY_LIFT_RANGE_CROSSES_ZERO, SAME_OBJECTIVE_PEERS_INSUFFICIENT, FALLBACK_TO_SHARED_PRIMARY_KPI_GROUP, FALLBACK_ACTION_CAPPED_AT_TEST, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND.

### Awareness Boost January
- **Action:** `KEEP_AS_TEST`; statistical decision `HOLD`.
- **Evidence:** 13033 successes from 1089878 eligible trials; raw 1.2%.
- **Uncertainty:** probability better 12.2%; lift range [-0.1%, +0.0%].
- **Benchmark:** \`shared_primary_kpi_group\` with \`provisional\` quality; portfolio context 1.1% is descriptive only.
- **Efficiency:** cpm is `worse_than_peer`.
- **Budget:** 1.19 units units; priority is 12.2% x 12.6%.
- **Conversation context:** Semantic coverage is 100.0%; the leading barrier is product_fit and the leading value driver is quality.
- **Reason codes:** PRIMARY_LIFT_RANGE_CROSSES_ZERO, SAME_OBJECTIVE_PEERS_INSUFFICIENT, FALLBACK_TO_SHARED_PRIMARY_KPI_GROUP, FALLBACK_ACTION_CAPPED_AT_TEST, BUDGET_WEIGHTED_BY_PRIMARY_PROBABILITY_X_SEMANTIC_LOWER_BOUND.

## 14. Objective Summary For Stakeholders

This final view keeps three audiences aligned:

- **Business owner:** allocated and unallocated budget, outcome quality, and economics.
- **Marketing director:** objective achievement, portfolio evidence, and strategic tests.
- **Performance marketing manager:** campaign, adset, ad, creative, and audience actions.

In [21]:
action_counts = campaigns["recommended_action"].value_counts().to_dict()
objective_budget = sum(
    allocations[str(campaign_id)].recommended_budget_units
    for campaign_id in campaigns["campaign_id"]
)
objective_spend_share = campaigns["spend"].sum() / result.scorecards["campaign"]["spend"].sum()
objective_envelope = policy.budget_units * objective_spend_share
objective_unallocated = objective_envelope - objective_budget
semantic_coverage = (
    campaigns["semantic_conversations"].sum()
    / campaigns["observed_conversations"].sum()
    if campaigns["observed_conversations"].sum() else np.nan
)

summary = pd.DataFrame([
    ["Campaigns", len(campaigns)],
    ["Final actions", action_counts],
    ["Previous-cycle objective spend share", objective_spend_share],
    ["Objective budget envelope", objective_envelope],
    ["Allocated to this objective", objective_budget],
    ["Unallocated in this objective", objective_unallocated],
    ["Conversation semantic coverage", semantic_coverage],
    ["Funded named tests", len(objective_tests)],
], columns=["Summary item", "Value"])
display(summary.style.hide(axis="index"))

display(Markdown(
    "**Decision discipline:** keep final actions as range-based reporting labels, "
    "allocate this POC's full objective envelopes with the explicit two-factor "
    "priority, and interpret overlapping ranges as uncertain rather than proven "
    "superiority."
))

Summary item,Value
Campaigns,2
Final actions,{'keep_as_test': 2}
Previous-cycle objective spend share,0.037222
Objective budget envelope,3.722161
Allocated to this objective,3.722161
Unallocated in this objective,0.000000
Conversation semantic coverage,1.000000
Funded named tests,2


**Decision discipline:** keep final actions as range-based reporting labels, allocate this POC's full objective envelopes with the explicit two-factor priority, and interpret overlapping ranges as uncertain rather than proven superiority.